# Redes Neurais Artificiais 2026.2

- **Disciplina**: Redes Neurais Artificiais 2026.2
- **Professora**: Elloá B. Guedes (ebgcosta@uea.edu.br)  
- **Github**: http://github.com/elloa  
        

Levando em conta a base de dados _Dry Bean Dataset_, esta parte do Projeto Prático diz respeito à proposição e avaliação de múltiplas redes neurais artificiais do tipo feedforward multilayer perceptron para o problema da classificação multiclasse de variedades de feijão seco a partir de características geométricas extraídas de imagens dos grãos.

## Busca em Grade

Uma maneira padrão de escolher os parâmetros de um modelo de Machine Learning é por meio de uma busca em grade via força bruta. O algoritmo da busca em grade é dado como segue:

1. Escolha a métrica de desempenho que você deseja maximizar (por exemplo, o F-Score com média macro, adequado a um problema multiclasse com classes desbalanceadas)  
2. Escolha o algoritmo de Machine Learning (exemplo: redes neurais artificiais). Em seguida, defina os parâmetros ou hiperparâmetros deste tipo de modelo sobre os quais você deseja otimizar (número de épocas, taxa de aprendizado, etc.) e construa um array de valores a serem testados para cada parâmetro ou hiperparâmetro.  
3. Defina a grade de busca, a qual é dada como o produto cartesiano dos valores de cada parâmetro a ser testado. Por exemplo, para os arrays [50, 100, 1000] e [10, 15], tem-se que a grade é [(50,10), (50,15), (100,10), (100,15), (1000,10), (1000,15)].
4. Para cada combinação de parâmetros a serem otimizados, utilize _apenas o conjunto de treinamento_ para realizar uma validação cruzada (holdout ou k-fold) e calcule a métrica de avaliação nos conjuntos de validação gerados por ela. O conjunto de teste não participa da busca. Vamos efetuar validação cruzada holdout 70/30 estratificada.
5. Escolha a combinação de parâmetros que maximiza a métrica de avaliação na validação. Este é o modelo otimizado, que deve então ser reavaliado uma única vez no conjunto de teste.

Por que esta abordagem funciona? Porque a busca em grade efetua uma pesquisa extensiva sobre as possíveis combinações de valores para cada um dos parâmetros a serem ajustados. Para cada combinação, ela estima a performance do modelo em dados não utilizados no seu treinamento (os conjuntos de validação). Por fim, o modelo com melhor métrica de desempenho é escolhido. Tem-se então que este modelo é o que tem maior chance de generalizar bem mediante dados nunca antes vistos, e o conjunto de teste, que não foi utilizado na busca, fornece a estimativa final e não enviesada desse desempenho.

## Efetuando a Busca em Grade sobre Hiperparâmetros das Top-6 RNAs

Considerando a etapa anterior do projeto prático, foram identificadas pelo menos 6 melhores Redes Neurais para o problema da classificação multiclasse das variedades de feijão no conjunto de dados selecionado. Estas redes podem ter sido obtidas pela estratégia própria ou pela regra da Pirâmide Geométrica, e todas utilizam os 16 atributos numéricos como preditores.

A primeira etapa desta segunda parte do projeto consiste em trazer para este notebook estas seis arquiteturas, ressaltando:

1. Número de neurônios ocultos por camada  
2. Função de Ativação  
3. Estratégia de origem da arquitetura (Própria ou Pirâmide Geométrica)  
4. Desempenho médio ± desvio padrão nos testes anteriores  
5. Número de repetições que a equipe conseguiu realizar para verificar os resultados  

Elabore uma busca em grade sobre estas arquiteturas que contemple variações nos hiperparâmetros a seguir, conforme documentação de [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

A. Solver (não usar o LBFGS, pois é mais adequado para datasets pequenos)  
B. Batch Size  
C. Learning Rate Init  
D. Paciência 
E. Épocas  

Nesta busca em grade, contemple a utilização do objeto [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), com a métrica escolhida no parâmetro `scoring` (por exemplo, `'f1_macro'`) e a validação cruzada realizada apenas com o conjunto de treinamento (já escalonado conforme a estratégia da etapa anterior).

In [1]:
# Importação de bibliotecas
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Silenciando avisos de não convergência durante a busca em grade
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Validação Cruzada k-fold

Na elaboração da busca em grade, vamos avaliar os modelos propostos segundo uma estratégia de validação cruzada ainda não explorada até o momento: a validação cruzada *k-fold*. Segundo a mesma, o conjunto de dados é particionado em *k* partes (*folds*): a cada iteração, separa-se uma das partes para validação e o modelo é treinado com as *k-1* partes remanescentes. Valores sugestivos de *k* na literatura são *k* = 3, 5 ou 10, pois o custo computacional desta validação dos modelos é alto. A métrica de desempenho é resultante da média dos desempenhos nas *k* iterações. A figura a seguir ilustra a ideia desta avaliação:

<img src="https://ethen8181.github.io/machine-learning/model_selection/img/kfolds.png" width="600">

Considerando a métrica de desempenho F1-Score (com média macro, dado que o problema possui 7 classes), considere a validação cruzada 5-fold estratificada para aferir os resultados da busca em grade anterior, realizada *apenas com o conjunto de treinamento*. O conjunto de teste deve ser utilizado uma única vez, para avaliar a melhor configuração encontrada.

In [ ]:
# 1. Carregando a base de dados Dry Bean Dataset (UCI ID = 602)
dry_bean = fetch_ucirepo(id=602)

X = dry_bean.data.features
y = dry_bean.data.targets["Class"]

# 2. Partição Holdout 70/30 estratificada (mantendo random_state=42 da etapa anterior)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# 3. Escalonamento dos atributos ajustado APENAS no conjunto de treinamento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Total da base:     X = {X.shape} | y = {y.shape}")
print(f"Conjunto de Treino: X_train_scaled = {X_train_scaled.shape} | y_train = {y_train.shape}")
print(f"Conjunto de Teste:  X_test_scaled  = {X_test_scaled.shape}  | y_test  = {y_test.shape}")

In [ ]:
# Top-6 arquiteturas distintas selecionadas da PP2.2.2 (3 da Estratégia Própria + 3 da Pirâmide Geométrica)
top6_arquiteturas = [
    {
        "ID": "RNA 1",
        "Camadas Ocultas": (32,),
        "Nº de Camadas": 1,
        "Função de Ativação": "relu",
        "Estratégia de Origem": "Pirâmide Geométrica (α = 3)",
        "F1-Score Anterior (Média ± DP)": "0.9435 ± 0.0041",
        "Acurácia Anterior (Média ± DP)": "0.9327 ± 0.0046",
        "Nº de Repetições": 10
    },
    {
        "ID": "RNA 2",
        "Camadas Ocultas": (30,),
        "Nº de Camadas": 1,
        "Função de Ativação": "relu",
        "Estratégia de Origem": "Estratégia Própria",
        "F1-Score Anterior (Média ± DP)": "0.9433 ± 0.0043",
        "Acurácia Anterior (Média ± DP)": "0.9323 ± 0.0048",
        "Nº de Repetições": 10
    },
    {
        "ID": "RNA 3",
        "Camadas Ocultas": (20, 12),
        "Nº de Camadas": 2,
        "Função de Ativação": "tanh",
        "Estratégia de Origem": "Pirâmide Geométrica (α = 3)",
        "F1-Score Anterior (Média ± DP)": "0.9426 ± 0.0044",
        "Acurácia Anterior (Média ± DP)": "0.9322 ± 0.0047",
        "Nº de Repetições": 10
    },
    {
        "ID": "RNA 4",
        "Camadas Ocultas": (30, 20),
        "Nº de Camadas": 2,
        "Função de Ativação": "relu",
        "Estratégia de Origem": "Estratégia Própria",
        "F1-Score Anterior (Média ± DP)": "0.9424 ± 0.0036",
        "Acurácia Anterior (Média ± DP)": "0.9315 ± 0.0041",
        "Nº de Repetições": 10
    },
    {
        "ID": "RNA 5",
        "Camadas Ocultas": (20, 10),
        "Nº de Camadas": 2,
        "Função de Ativação": "tanh",
        "Estratégia de Origem": "Estratégia Própria",
        "F1-Score Anterior (Média ± DP)": "0.9417 ± 0.0038",
        "Acurácia Anterior (Média ± DP)": "0.9310 ± 0.0042",
        "Nº de Repetições": 10
    },
    {
        "ID": "RNA 6",
        "Camadas Ocultas": (21,),
        "Nº de Camadas": 1,
        "Função de Ativação": "relu",
        "Estratégia de Origem": "Pirâmide Geométrica (α = 2)",
        "F1-Score Anterior (Média ± DP)": "0.9415 ± 0.0038",
        "Acurácia Anterior (Média ± DP)": "0.9306 ± 0.0040",
        "Nº de Repetições": 10
    }
]

df_top6 = pd.DataFrame(top6_arquiteturas)
display(df_top6)

In [ ]:
# Configuração da Validação Cruzada 5-Fold Estratificada (pro GridSearchCV)
cv_5fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Exemplo de dicionário param_grid contemplando as 6 arquiteturas e os itens A, B, C, D e E:
# (Como cada arquitetura já possui sua função de ativação fixa da etapa anterior,
# usamos uma lista de dicionários para parear cada hidden_layer_sizes com sua respectiva activation)
grade_parametros_base = {
    "solver": ["adam", "sgd"],               # A. Solver (sem lbfgs)
    "batch_size": [64, 200],                 # B. Batch Size
    "learning_rate_init": [0.001, 0.01],     # C. Learning Rate Init
    "n_iter_no_change": [10, 20],            # D. Paciência
    "max_iter": [150, 250]                   # E. Épocas
}

param_grid = [
    {
        "hidden_layer_sizes": [arq["Camadas Ocultas"]],
        "activation": [arq["Função de Ativação"]],
        **grade_parametros_base
    }
    for arq in top6_arquiteturas
]

# ATENÇÃO AQUIIIIIII: basta passar n_jobs=-1 no GridSearchCV para rodar em todos os núcleos da CPU!
# grid_search = GridSearchCV(
#     estimator=MLPClassifier(random_state=42),
#     param_grid=param_grid,
#     scoring="f1_macro",
#     cv=cv_5fold,
#     n_jobs=-1,
#     verbose=1
# )
# grid_search.fit(X_train_scaled, y_train)

## Identificando a mellhor solução

Como resultado da busca em grade com validação cruzada 5-fold, identifique o modelo otimizado com melhor desempenho para o problema. Apresente claramente este modelo, sua arquitetura (número de camadas ocultas, neurônios por camada e função de ativação), os hiperparâmetros otimizados (solver, batch size, taxa de aprendizado inicial, paciência e épocas) e os resultados (F1-Score macro) para cada um dos folds avaliados, junto com a média ± desvio padrão entre os folds. Em seguida, reavalie este modelo uma única vez no conjunto de teste, apresentando a matriz de confusão, a acurácia e o F1-Score macro. Esta é a melhor solução identificada em decorrência deste projeto.

## Empacotando a solução

Suponha que você deve entregar este classificador a uma cooperativa ou órgão responsável pelo controle de qualidade e classificação de grãos de feijão seco. Para tanto, você deve fazer uma preparação do mesmo para utilização neste cenário. Uma vez que já identificou os melhores hiperparâmetros, o passo remanescente consiste em treinar o modelo com estes valores e todos os dados disponíveis, salvando o modelo treinado (pesos da rede e o escalonador dos atributos) ao final para entrega ao cliente. Assim, finalize o projeto prático realizando tais passos.

1. Consulte a documentação a seguir:
https://scikit-learn.org/stable/modules/model_persistence.html  
2. Treine o modelo com todos os dados, utilizando a arquitetura e os hiperparâmetros otimizados. O escalonador (`StandardScaler`) deve ser ajustado com todos os dados e salvo junto com a rede, preferencialmente em um único `Pipeline`  
3. Salve o modelo em disco  
4. Construa uma rotina que recupere o modelo em disco  
5. Mostre que a rotina é funcional, fazendo previsões com todos os elementos do dataset e exibindo uma matriz de confusão das mesmas. Como o modelo foi treinado com todos os dados, esta matriz serve apenas para verificar o funcionamento da rotina e não como estimativa de desempenho, que é dada pelos resultados da validação cruzada e do conjunto de teste